RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
#import all the documents required for completing pipeline
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

d:\RAG AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Read all the pdfs inside the directory, based on the workspace locatio we are going to get the pdf path. length of file, printing pdf file
def process_all_pdfs(pdf_directory):
    """process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF Files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" ✔ Loaded {len(documents)} pages")

        except Exception as e:
            print(f" X Error: (e)")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
#process all pdfs in the data directory
all_pdf_documents = process_all_pdfs("../data")                    

Found 4 PDF Files to process

Processing: Agarbatti making Project KVIC MSME PROGRAM.pdf
 ✔ Loaded 22 pages

Processing: flower Agarbatti research.pdf
 ✔ Loaded 9 pages

Processing: FLOWER WASTE INTO INCENSEIJISRT25NOV303.pdf
 ✔ Loaded 6 pages

Processing: Green+Solution+For+Sacred+Waste_+Developing+Eco-Friendly+Aromatic+Product.pdf
 ✔ Loaded 13 pages

Total documents loaded: 50


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-09-16T12:03:34+00:00', 'author': 'MSME', 'moddate': '2020-09-16T12:03:34+00:00', 'source': '..\\data\\pdf\\Agarbatti making Project KVIC MSME PROGRAM.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'source_file': 'Agarbatti making Project KVIC MSME PROGRAM.pdf', 'file_type': 'pdf'}, page_content='KHADI AND VILLAGE INDUSTRIES COMMISSION \n(Ministry of MSME ) \n \nOPERATIONAL GUIDELINES FOR THE PILOT PROJECTS OF AGARBATTI \nINDUSTRY UNDER POLYMER AND CH EMICAL BASED INDUSTR Y(PCBI ) \nVERTICAL OF GRAMODYOG VIKAS YOJANA(GVY) \n \n1.  Preamble   \n \nAgarbatti making is a traditional industry in India with a size of around Rs. 7500 \ncrore annual production with involvement of about 5 lakh people and export of about Rs. \n750 crore s.  Based on the interaction with industry on 20th August 2020, it came out \nthat, today the industry is facing problem in the area of raw materi

In [4]:
#text splitting getting started with chunks- for chunking use the recursivecharatcertext splitting

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #shoing example of a chunk
    if split_docs:
        print(f"\nExample Chunk:")
        print(f"content: {split_docs[0].page_content[:200]}...")
        print(f"metadata: {split_docs[0].metadata}")

    return split_docs    

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 50 documents into 201 chunks

Example Chunk:
content: KHADI AND VILLAGE INDUSTRIES COMMISSION 
(Ministry of MSME ) 
 
OPERATIONAL GUIDELINES FOR THE PILOT PROJECTS OF AGARBATTI 
INDUSTRY UNDER POLYMER AND CH EMICAL BASED INDUSTR Y(PCBI ) 
VERTICAL OF GRA...
metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-09-16T12:03:34+00:00', 'author': 'MSME', 'moddate': '2020-09-16T12:03:34+00:00', 'source': '..\\data\\pdf\\Agarbatti making Project KVIC MSME PROGRAM.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'source_file': 'Agarbatti making Project KVIC MSME PROGRAM.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-09-16T12:03:34+00:00', 'author': 'MSME', 'moddate': '2020-09-16T12:03:34+00:00', 'source': '..\\data\\pdf\\Agarbatti making Project KVIC MSME PROGRAM.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'source_file': 'Agarbatti making Project KVIC MSME PROGRAM.pdf', 'file_type': 'pdf'}, page_content='KHADI AND VILLAGE INDUSTRIES COMMISSION \n(Ministry of MSME ) \n \nOPERATIONAL GUIDELINES FOR THE PILOT PROJECTS OF AGARBATTI \nINDUSTRY UNDER POLYMER AND CH EMICAL BASED INDUSTR Y(PCBI ) \nVERTICAL OF GRAMODYOG VIKAS YOJANA(GVY) \n \n1.  Preamble   \n \nAgarbatti making is a traditional industry in India with a size of around Rs. 7500 \ncrore annual production with involvement of about 5 lakh people and export of about Rs. \n750 crore s.  Based on the interaction with industry on 20th August 2020, it came out \nthat, today the industry is facing problem in the area of raw materi

#Embedding and vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer 
import chromadb            
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 455.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


C:\Users\shiva\AppData\Local\Temp\ipykernel_21996\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


###Vector Store

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 402


In [9]:
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-09-16T12:03:34+00:00', 'author': 'MSME', 'moddate': '2020-09-16T12:03:34+00:00', 'source': '..\\data\\pdf\\Agarbatti making Project KVIC MSME PROGRAM.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'source_file': 'Agarbatti making Project KVIC MSME PROGRAM.pdf', 'file_type': 'pdf'}, page_content='KHADI AND VILLAGE INDUSTRIES COMMISSION \n(Ministry of MSME ) \n \nOPERATIONAL GUIDELINES FOR THE PILOT PROJECTS OF AGARBATTI \nINDUSTRY UNDER POLYMER AND CH EMICAL BASED INDUSTR Y(PCBI ) \nVERTICAL OF GRAMODYOG VIKAS YOJANA(GVY) \n \n1.  Preamble   \n \nAgarbatti making is a traditional industry in India with a size of around Rs. 7500 \ncrore annual production with involvement of about 5 lakh people and export of about Rs. \n750 crore s.  Based on the interaction with industry on 20th August 2020, it came out \nthat, today the industry is facing problem in the area of raw materi

In [10]:
#convert the text to embeddings
texts=[doc.page_content for doc in chunks]

##Generate the embeddings

embeddings=embedding_manager.generate_embeddings(texts)

#store in the vector detabase
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 201 texts...


Batches: 100%|██████████| 7/7 [01:16<00:00, 10.87s/it]


Generated embeddings with shape: (201, 384)
Adding 201 documents to vector store...
Successfully added 201 documents to vector store
Total documents in collection: 603


RAG retriever Pipeline from VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("WHat is the process of making agarbatti")

Retrieving documents for query: 'WHat is the process of making agarbatti'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.29it/s]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_ba163a70_1',
  'content': 'for Agarbatti making. While these raw materials may not constitute major cost towards \nmaking of Agarbatti (about 20% by these raw materials) but their indigenous production \nwill provide good opportunity to create more jobs and also to assure regu lar supply of \nraw materials at  reasonable rate s. This, in the long term , will help keep the price of \nfinished goods reasonably stable. Other important area to look into, includes inputs on  \n‘fragrance’ and the machines manufacturing capability in the country for making \nAgarbatti. India is strong in the ‘fragrance’ portion but needs to develop strength in \nmachine making which is largely imported, presently. Accordingly a holistic approach is \nrequired to strengthen this traditional industry so as to  sustain the present growth rate \nof about 10%, and to capture more and more export markets.   \nThe major components of this holistic approach will be to address issues of',
  'metadata': {

In [14]:
rag_retriever.retrieve("what are the key features of organic incense")

Retrieving documents for query: 'what are the key features of organic incense'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.73it/s]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e29e0e47_76',
  'content': 'Usage 0.28 0.09 3.11 0.003\nAromatic properties 0.43 0.12 3.57 0.001\nCost \x00 0.21 0.07 \x00 2.95 0.005\nHealth benefits 0.38 0.11 3.46 0.002\nMarket potential 0.19 0.08 2.35 0.023\nEntrepreneur scope 0.15 0.06 2.56 0.015\nTable 2c \nPost Hoc Test (Tukey HSD).\nComparison Mean Difference p-value\nSynthetic Brand vs. Organic Incense 1 \x00 17.50 0.002\nSynthetic Brand vs. Organic Incense 2 \x00 17.75 0.001\nOrganic Incense 1 vs. Organic Incense 2 \x00 0.25 0.943\nM. Sekar and M. Dhanraj                                                                                                                                                                                                                     Cleaner Waste Systems 12 (2025) 100426 \n4',
  'metadata': {'producer': 'Acrobat Distiller 8.1.0 (Windows)',
   'creationdate': '2025-12-08T20:45:07+00:00',
   'robots': 'noindex',
   'crossmarkdomains[1]': 'elsevier.com',
   'creationdate--text': '9th D

Integration VectorDB context pipeline with LLM output

In [ ]:
###simple RAG Pipeline with Groq LLM
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()

##initialize the Groq LLM ( set your GROQ_API_KEY in environment)
groq_api_key = os.getenv(GROQ_API_KEY)

##2. Simple RAG function: Retrieve context + generate response
def rag_simple(query, retriever,llm,top+k=3):
    ## retrieve the context
    results=retiever.retrieve(query,k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## Generate the answer using Groq LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:""" 
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


SyntaxError: invalid syntax (2161321795.py, line 11)

In [ ]:
answer=rag_simple("Explain me about the skill development program by kvic")
print(answer)

Enhanced RAG Pipeline Features

In [ ]:
#----Enhanced RAG Pipeling----
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    ## Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300]+ '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{Context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.context,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output
    
# Example Usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])    

In [ ]:
# ---Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = [] #Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retriever relevant documents
        results = self.retriever.retrieve(question, top_k=top_k,score_threshold=min_score)
        if not results:
            answer = "No relevant context found"
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page' : doc['metadata'].get('page', 'unknown'),
                'score' : doc['similarity_score'],
                'preview' : doc['content'][:120] + '...'
            } for doc in results]

            #streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.
             \nContext:\n{context}\n\nQuestion: {question}\n\nAnswer: """
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source]} (page {src['page]})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        #Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # store query history
        self.history.append({
            'question' : question,
            'answer' : answer,
            'sources' : sources,
            'summary' : summary,
            'history' : self.history
        })         

        return {
            'question' : question,
            'answer' : answer_with_citations,
            'sources' : sources,
            'summary' : summary,
            'history' : self.history
        }        
    
    # Example usage:
    adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
    result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
    print("\nFinal Answer:", result['answer'])
    print("summary:", result['summary'])
    print("History:", result['history'][-1])